This example explains how to use a Pretained embeddings from Glove for Sentiment Analysis
The corpus is created using Word2Vec. This corpus is provided as input. 

In [3]:
from numpy import array

from tensorflow.keras.preprocessing.text import one_hot
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import Flatten
from tensorflow.keras.layers import Embedding


In [4]:
corpus = [
    # Positive Reviews

    'This is an excellent movie',
    'The move was fantastic I like it',
    'You should watch it is brilliant',
    'Exceptionally good',
    'Wonderfully directed and executed I like it',
    'Its a fantastic series',
    'Never watched such a brillent movie',
    'It is a Wonderful movie puttenalli',

    # Negtive Reviews

    "horrible acting",
    'waste of money',
    'pathetic picture',
    'It was very boring',
    'I did not like the movie',
    'The movie was horrible',
    'I will not recommend',
    'The acting is pathetic'
]

In [5]:
sentiments = array([1,1,1,1,1,1,1,1,0,0,0,0,0,0,0,0])

In [8]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [7]:
word_tokenizer = Tokenizer()
word_tokenizer.fit_on_texts(corpus)

In [9]:
vocab_length = len(word_tokenizer.word_index) + 1
vocab_length

45

convert sentences to their numeric counterpart, call the texts_to_sequences function and pass it the whole corpus.

In [10]:
# Tokenizing the sentences by assigning unique integers to each word in the corpus. 
# These are not word embeddings, just unique integers assigned to each word.
tokenized_sentences = word_tokenizer.texts_to_sequences(corpus)
tokenized_sentences

[[14, 3, 15, 16, 1],
 [4, 17, 6, 9, 5, 7, 2],
 [18, 19, 20, 2, 3, 21],
 [22, 23],
 [24, 25, 26, 27, 5, 7, 2],
 [28, 8, 9, 29],
 [30, 31, 32, 8, 33, 1],
 [2, 3, 8, 34, 1, 35],
 [10, 11],
 [36, 37, 38],
 [12, 39],
 [2, 6, 40, 41],
 [5, 42, 13, 7, 4, 1],
 [4, 1, 6, 10],
 [5, 43, 13, 44],
 [4, 11, 3, 12]]

find the number of words in the longest sentence and then to apply padding to the sentences having shorter lengths than the length of the longest sentence.

In [11]:
import nltk
#nltk.download('popular')
nltk.download('punkt')
nltk.download('punkt_tab')


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\reach\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\reach\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [12]:
from nltk.tokenize import word_tokenize

word_count = lambda sentence: len(word_tokenize(sentence))

longest_sentence     = max(corpus, key=word_count)
length_long_sentence = len(word_tokenize(longest_sentence))

padded_sentences = pad_sequences(tokenized_sentences, length_long_sentence, padding='post')

print(padded_sentences)

[[14  3 15 16  1  0  0]
 [ 4 17  6  9  5  7  2]
 [18 19 20  2  3 21  0]
 [22 23  0  0  0  0  0]
 [24 25 26 27  5  7  2]
 [28  8  9 29  0  0  0]
 [30 31 32  8 33  1  0]
 [ 2  3  8 34  1 35  0]
 [10 11  0  0  0  0  0]
 [36 37 38  0  0  0  0]
 [12 39  0  0  0  0  0]
 [ 2  6 40 41  0  0  0]
 [ 5 42 13  7  4  1  0]
 [ 4  1  6 10  0  0  0]
 [ 5 43 13 44  0  0  0]
 [ 4 11  3 12  0  0  0]]


In [14]:
from numpy import array
from numpy import asarray
from numpy import zeros

embeddings_dictionary = dict()

glove_file = open(r'D:\Makesh\Working\AI\RPS\Day04\Dataset\glove\archive\glove.6B.100d.txt', encoding="utf8")

We will create a dictionary that will contain words as keys and the corresponding 100 dimensional vectors as values, in the form of an array.

In [15]:
for line in glove_file:
    
    records = line.split()
    word    = records[0]
    
    vector_dimensions = asarray(records[1:], dtype='float32')
    
    embeddings_dictionary [word] = vector_dimensions

glove_file.close()

The dictionary embeddings_dictionary now contains words and corresponding GloVe embeddings for all the words.

In [16]:
embedding_matrix = zeros((vocab_length, 100))

for word, index in word_tokenizer.word_index.items():
    
    embedding_vector = embeddings_dictionary.get(word)
    
    if embedding_vector is not None:
        embedding_matrix[index] = embedding_vector

In [ ]:
embedding_matrix[:5]

embedding_matrix now contains pretrained word embeddings for the words in our corpus.

In [19]:
model = Sequential()
embedding_layer = Embedding(input_dim    =vocab_length, 
                            output_dim   =100, 
                            weights      =[embedding_matrix], 
                            input_length =length_long_sentence, 
                            trainable    =False)
model.add(embedding_layer)
model.add(Flatten())
model.add(Dense(1, activation='sigmoid'))

d:\Makesh\Working\AI\Python_Envs\gen_ai\Lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


- weights parameter. You can pass your __pretrained embedding matrix__ as default weights to the weights parameter. 
- since we are not training the embedding layer, the __trainable__ attribute has been set to __False__.

In [20]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['acc'])
print(model.summary())

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │         4,500 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,500 (17.58 KB)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 4,500 (17.58 KB)

None


You can see that since we have 44 words in our vocabulary and each word will be represented as a 100 dimensional vector, the number of parameters for the embedding layer will be 44 x 100 = 4400. The output from the embedding layer will be a 2D vector with 7 rows (1 for each word in the sentence) and 100 columns. The output from the embedding layer will be flattened so that it can be used with the dense layer. Finally the dense layer is used to make predictions.

In [21]:
model.fit(padded_sentences, sentiments, epochs=100, verbose=1)

Epoch 1/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 381ms/step - acc: 0.4375 - loss: 0.7595
Epoch 2/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - acc: 0.4375 - loss: 0.7334
Epoch 3/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - acc: 0.5000 - loss: 0.7083
Epoch 4/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step - acc: 0.5000 - loss: 0.6841
Epoch 5/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - acc: 0.5000 - loss: 0.6609
Epoch 6/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - acc: 0.5625 - loss: 0.6386
Epoch 7/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - acc: 0.5625 - loss: 0.6172
Epoch 8/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - acc: 0.5625 - loss: 0.5966
Epoch 9/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - acc: 0.5625 - loss: 0.5768
Epoch 10/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - acc: 0.6875 - loss: 0.5577
Epoch 11/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - acc: 0.6875 - loss: 0.5393
Epoch 12/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - acc: 0.7500 - loss: 0.5216
Epoch 13/100
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/st

In [22]:
loss, accuracy = model.evaluate(padded_sentences, sentiments, verbose=0)
print('Accuracy: %f' % (accuracy*100))

Accuracy: 100.000000


Since model uses all the train data, accuracy is very high ie(100%). To find the real accuracy, lets split the input data like below

In [23]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    padded_sentences, sentiments, test_size=0.2, random_state=42
)

In [24]:
model.fit(X_train, y_train, epochs=10, batch_size=32)

Epoch 1/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - acc: 1.0000 - loss: 0.0813
Epoch 2/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - acc: 1.0000 - loss: 0.0801
Epoch 3/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - acc: 1.0000 - loss: 0.0790
Epoch 4/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - acc: 1.0000 - loss: 0.0778
Epoch 5/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - acc: 1.0000 - loss: 0.0766
Epoch 6/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - acc: 1.0000 - loss: 0.0754
Epoch 7/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 27ms/step - acc: 1.0000 - loss: 0.0742
Epoch 8/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 31ms/step - acc: 1.0000 - loss: 0.0730
Epoch 9/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 29ms/step - acc: 1.0000 - loss: 0.0718
Epoch 10/10
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - acc: 1.0000 - loss: 0.0707


In [25]:
loss, accuracy = model.evaluate(X_test, y_test, verbose=0)
print(f"Test Accuracy = {accuracy * 100:.2f}%")

Test Accuracy = 100.00%


Inference the model with new data

In [47]:
# 1. Your new text
new_text = ["fantastic series with brilliant acting and direction"]

# 2. Convert text to sequence using the SAME tokenizer used for training
seq = word_tokenizer.texts_to_sequences(new_text)

# 3. Pad using the SAME length used during training
padded = pad_sequences(seq, maxlen=length_long_sentence, padding='post')

# 4. Predict sentiment
prediction = model.predict(padded)

print("Raw prediction:", prediction)
print("Predicted sentiment:", "Positive" if prediction[0][0] > 0.5 else "Negative")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step
Raw prediction: [[0.53818864]]
Predicted sentiment: Positive
